# Run This Project on Google Colab (Research Only)

Trains / evaluates the Jointly Optimized PPO Portfolio Management System (regime encoder + actor-critic PPO agent) on CSE market data, using `main.py` from the repo.

This notebook runs **only the research pipeline** (`src/`, `main.py`) — the `backend/` (FastAPI) and `frontend/` (React) web-demo apps are not needed and are skipped entirely.

**Before you start:**
1. `Runtime > Change runtime type > T4 GPU` (or better) — training uses PyTorch and is much faster on GPU.
2. `data/raw/` is excluded from the git repo (see `.gitignore`), so you need to make the raw CSE CSV files available to Colab yourself. Two ways, both handled below:
   - Upload the `data/raw/` folder to your Google Drive once, then just mount Drive here (fast, repeatable).
   - Or upload the CSV files directly into this Colab session with the file picker (quick one-off runs, lost when the runtime resets).

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Get the project code

Clones the public repo. If you have local changes not pushed to GitHub yet, push them first (or use the Drive-sync alternative in the commented-out cell below).

In [ ]:
REPO_URL = "https://github.com/sathsarasithum/portfolio_regime_detection.git"
PROJECT_DIR = "/content/portfolio_regime_detection"

!git clone -q {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

# Research-only run: drop the web-demo apps, not needed here
!rm -rf backend frontend

In [ ]:
# Alternative: instead of cloning from GitHub, sync the project from Google Drive
# (use this if you have uncommitted local changes you want to run as-is).
#
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_DIR = "/content/drive/MyDrive/Regime detection portfolio"  # edit to match your Drive path
# %cd {PROJECT_DIR}

## 2. Install dependencies

Colab already ships a CUDA-enabled `torch`/`torchvision`, so we skip those two lines from `requirements.txt` to avoid pip overwriting them with a mismatched build. We also skip the web-demo-only packages (`fastapi`, `uvicorn`, `joblib`, `requests`) since `backend/`/`frontend/` aren't part of this run.

In [ ]:
from pathlib import Path

SKIP_PREFIXES = (
    "torch>", "torch=", "torch ", "torchvision",          # Colab already has a CUDA build
    "fastapi", "uvicorn", "joblib", "requests",            # backend/ web-demo only, not needed for research
)

req_lines = Path("requirements.txt").read_text().splitlines()
filtered = [
    line for line in req_lines
    if not line.strip().lower().startswith(SKIP_PREFIXES)
]
Path("requirements.colab.txt").write_text("\n".join(filtered))

!pip install -q -r requirements.colab.txt

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Provide the data (`data/raw/`)

Expected files (CSE banking-sector price/volume CSVs), e.g. `2016_banking_sector.csv` ... `2025_banking_sector.csv`.

**Option A — from Google Drive (recommended):** upload your local `data/raw/` folder to Drive once, e.g. to `MyDrive/Regime detection portfolio/data/raw/`, then set `DRIVE_DATA_PATH` below and run this cell.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

DRIVE_DATA_PATH = "/content/drive/MyDrive/Regime detection portfolio/data/raw"  # edit to match your Drive path

if os.path.isdir(DRIVE_DATA_PATH):
    shutil.copytree(DRIVE_DATA_PATH, "data/raw", dirs_exist_ok=True)
    print(f"Copied data from Drive: {DRIVE_DATA_PATH}")
else:
    print(f"Drive path not found: {DRIVE_DATA_PATH}\nUse the upload cell below instead, or fix the path and re-run.")

**Option B — one-off manual upload** (skip this cell if Option A already populated `data/raw/`):

In [ ]:
from google.colab import files
import os

os.makedirs("data/raw", exist_ok=True)
uploaded = files.upload()  # select your CSE raw CSV files
for fname in uploaded:
    os.replace(fname, os.path.join("data/raw", fname))

In [ ]:
raw_files = sorted(os.listdir("data/raw"))
assert len(raw_files) > 0, "data/raw is empty — populate it via Option A or B above before continuing."
print(f"{len(raw_files)} files in data/raw:")
raw_files

## 4. Configure the run

Mirrors the CLI flags in `main.py`. Lower `TOTAL_TIMESTEPS` for a quick smoke test, raise it for a real training run.

In [ ]:
MODE = "train"              # "train", "eval", or "backtest"
N_ASSETS = 47
TOTAL_TIMESTEPS = 10000
EPOCHS = 10
BATCH_SIZE = 64
ROLLOUT_LENGTH = 256
LEARNING_RATE = 3e-4
SEED = 42

## 5. Train / evaluate / backtest

In [ ]:
!python main.py \
  --mode {MODE} \
  --n-assets {N_ASSETS} \
  --total-timesteps {TOTAL_TIMESTEPS} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --rollout-length {ROLLOUT_LENGTH} \
  --lr {LEARNING_RATE} \
  --seed {SEED} \
  --device cuda

## 6. Inspect results

In [ ]:
import json

with open("results/test_summary.json") as f:
    summary = json.load(f)
summary

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weights_df = pd.read_csv("results/test_weights.csv")

plt.figure(figsize=(12, 5))
plt.plot(weights_df["step"], weights_df["portfolio_value"], color="#2196F3", linewidth=2)
plt.title("Portfolio Value — Test Set")
plt.xlabel("Trading Day")
plt.ylabel("Portfolio Value (LKR)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. (Optional) Save results back to Drive

The Colab runtime is ephemeral — copy `results/` and `experiments/` (checkpoints, logs, summaries) to Drive so they survive after the session ends.

In [ ]:
SAVE_TO_DRIVE = "/content/drive/MyDrive/Regime detection portfolio/colab_runs/run_01"  # edit as needed

os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
shutil.copytree("results", os.path.join(SAVE_TO_DRIVE, "results"), dirs_exist_ok=True)
shutil.copytree("experiments", os.path.join(SAVE_TO_DRIVE, "experiments"), dirs_exist_ok=True)
print(f"Saved to: {SAVE_TO_DRIVE}")